# Unir Feature_Station_2015 y SeisBenchV1_v1_1
Esta notebook carga los archivos `Feature_Station_2015.json` y `SeisBenchV1_v1_1.json`, inspecciona columnas comunes, intenta unirlos automáticamente (por columna común si existe, o concatenando columnas si no), y guarda el resultado en `merged_datasets.json`.

Asegúrate de que ambos archivos están en la misma carpeta que esta notebook (el root del workspace).

In [26]:
# Cargar librerías
import pandas as pd
from pathlib import Path
import json
import re

workspace = Path('.')
print('Workspace:', workspace.resolve())
print('pandas version:', pd.__version__)

Workspace: C:\Users\ricar\OneDrive\Desktop\TESIS
pandas version: 2.3.2


In [27]:
# Rutas de los archivos
f1 = workspace / 'Feature_Station_2015.json'
f2 = workspace / 'SeisBenchV1_v1_1.json'
print('Existe f1?', f1.exists())
print('Existe f2?', f2.exists())

# Intentar cargar como JSON estándar (si falla, mostrar error)
try:
    df1 = pd.read_json(f1)
except Exception as e:
    print('Error al leer', f1, '\n', e)
    df1 = None

try:
    df2 = pd.read_json(f2)
except Exception as e:
    print('Error al leer', f2, '\n', e)
    df2 = None

# Mostrar información básica
if df1 is not None:
    print('--- df1 head and shape ---')
    display(df1.head())
    print('df1.shape =', df1.shape)
    print('df1.info():', df1.info())
if df2 is not None:
    print('--- df2 head and shape ---')
    display(df2.head())
    print('df2.shape =', df2.shape)
    print('df2.info():', df2.info())

Existe f1? True
Existe f2? True
--- df1 head and shape ---


,Event_Identifier,Station,Channel,Type,t_mean,t_std,t_var,t_entropy,t_kurtosis,t_multiscaleEntropy,...,w_t_peak2peak_D5,w_t_peak2peak_D6,w_t_peak2rms_A6,w_t_peak2rms_D1,w_t_peak2rms_D2,w_t_peak2rms_D3,w_t_peak2rms_D4,w_t_peak2rms_D5,w_t_peak2rms_D6,w_t_meanEnergyCoeff
0,7e7ea43c735bbfe9e9c1fe73ed97a7e3,BNAS,BHZ,EXPL,-0.035686,0.249026,0.062014,152.217360,4.673847,1.902171,...,0.586886,0.214210,3.526992,4.767929,5.332523,4.040610,3.487185,3.086393,2.744577,0.074095
1,1cc34bbf0bd57cf025c065df41c0f1a1,BNAS,BHZ,EXPL,-0.042388,0.255646,0.065355,149.471451,4.180068,1.790757,...,0.421342,0.137214,1.757422,5.205819,4.271823,4.018401,3.607966,4.349338,2.768516,0.068003
2,d718b9eba8d1d2a1524a4e1e0888d2f8,BNAS,BHZ,EXPL,-0.103105,0.238584,0.056923,150.992826,4.036087,1.840276,...,0.363225,0.141073,1.306040,5.389804,5.227752,4.357040,4.479645,3.081091,3.684229,0.069561
3,3b1ecf423de57ed90c45bdb0194caa60,BNAS,BHZ,EXPL,-0.029458,0.210861,0.044462,108.191907,4.841379,1.799442,...,0.236537,0.087113,1.456827,5.716850,4.151468,3.436199,3.158368,3.396124,2.617087,0.047153
4,1941b6be00dc0edae7b852bfbf1124c3,BNAS,BHZ,EXPL,-0.089604,0.251170,0.063086,165.719234,3.963954,1.839883,...,0.130397,0.085275,1.542586,6.386649,4.776877,3.788521,3.581984,3.180676,2.856925,0.074423


df1.shape = (22640, 88)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22640 entries, 0 to 22639
Data columns (total 88 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Event_Identifier            22640 non-null  object 
 1   Station                     22640 non-null  object 
 2   Channel                     22640 non-null  object 
 3   Type                        22640 non-null  object 
 4   t_mean                      22640 non-null  float64
 5   t_std                       22640 non-null  float64
 6   t_var                       22640 non-null  float64
 7   t_entropy                   22640 non-null  float64
 8   t_kurtosis                  22640 non-null  float64
 9   t_multiscaleEntropy         22640 non-null  float64
 10  t_time2peak                 22640 non-null  float64
 11  t_rms                       22640 non-null  float64
 12  t_peak2peak                 22640 non-null  float64
 13  t_peak2

,Type,f1_t_mean,f2_t_std,f3_t_var,f4_t_entropy,f5_t_kurtosis,f6_t_multiscaleEntropy,f7_t_time2peak,f8_t_rms,f9_t_peak2peak,...,f75_w_t_peak2peak_D5,f76_w_t_peak2peak_D6,f77_w_t_peak2rms_A6,f78_w_t_peak2rms_D1,f79_w_t_peak2rms_D2,f80_w_t_peak2rms_D3,f81_w_t_peak2rms_D4,f82_w_t_peak2rms_D5,f83_w_t_peak2rms_D6,f84_w_t_meanEnergyCoeff
0,VT,-0.000059,0.159283,0.025371,125.575049,8.304542,1.659625,1.49,0.159248,1.735696,...,0.194575,0.028046,10.062697,8.860736,7.890076,6.719720,4.237517,3.802709,8.048929,0.024310
1,LP,-0.000042,0.197845,0.039143,393.853185,5.290367,2.004580,10.06,0.197824,1.837824,...,0.587130,0.051686,19.361546,5.857056,4.538726,4.670957,5.671383,4.123603,3.331997,0.038853
2,LP,0.000037,0.190926,0.036453,396.682354,6.666304,1.744531,14.87,0.190909,1.934064,...,1.221593,0.058025,10.923910,4.332089,4.554143,6.152058,4.511500,7.012052,5.524468,0.035855
3,VT,-0.000049,0.363053,0.131808,620.945134,3.580948,1.614374,11.93,0.363016,1.807268,...,0.937122,0.139498,4.845511,11.707501,7.343261,5.640008,4.287812,5.132921,4.930505,0.128911
4,VT,-0.000039,0.217804,0.047439,235.107723,6.647344,1.567582,3.76,0.217769,1.730968,...,0.347865,0.028109,10.195495,9.866074,8.192508,5.847093,4.932813,4.201841,5.791239,0.045848


df2.shape = (1187, 85)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1187 entries, 0 to 1186
Data columns (total 85 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Type                            1187 non-null   object 
 1   f1_t_mean                       1187 non-null   float64
 2   f2_t_std                        1187 non-null   float64
 3   f3_t_var                        1187 non-null   float64
 4   f4_t_entropy                    1187 non-null   float64
 5   f5_t_kurtosis                   1187 non-null   float64
 6   f6_t_multiscaleEntropy          1187 non-null   float64
 7   f7_t_time2peak                  1187 non-null   float64
 8   f8_t_rms                        1187 non-null   float64
 9   f9_t_peak2peak                  1187 non-null   float64
 10  f10_t_peak2rms                  1187 non-null   float64
 11  f11_t_energy                    1187 non-null   float64
 12  f12_t_zcr  

In [28]:
# Lógica para unir: limpiar df1, renombrar df2 con regex y concatenar verticalmente
if df1 is None or df2 is None:
    raise RuntimeError('Ambos dataframes deben cargarse correctamente para proceder')

# Eliminar columnas irrelevantes de df1
drop_cols = ['Event_Identifier', 'Station', 'channel']
df1 = df1.drop(columns=[c for c in drop_cols if c in df1.columns], errors='ignore')
print('Columnas de df1 tras eliminar irrelevantes:', list(df1.columns))

# Renombrar columnas de df2 eliminando solo el prefijo inicial f<digits>_ (por ejemplo f1_, f2_, f14_)
pattern = re.compile(r'^f\d+_(.+)$', re.IGNORECASE)
rename_map = {col: pattern.sub(r'\1', col) if pattern.match(col) else col for col in df2.columns}
df2 = df2.rename(columns=rename_map)
print('Renombrado de columnas df2:')
for original, new in rename_map.items():
    if original != new:
        print(f'  {original} -> {new}')

common = list(set(df1.columns) & set(df2.columns))
print('Columnas comunes tras renombrado:', sorted(common))
print('Columnas únicas en df1:', sorted(set(df1.columns) - set(df2.columns)))
print('Columnas únicas en df2:', sorted(set(df2.columns) - set(df1.columns)))

# Concatenar filas para unir ambos datasets una vez que las columnas se alinean
all_columns = sorted(set(df1.columns) | set(df2.columns))
merged = pd.concat([
    df1.reindex(columns=all_columns),
    df2.reindex(columns=all_columns)
], ignore_index=True)

# Eliminar la columna Channel del dataset unido si existe
merged = merged.drop(columns=['Channel'], errors='ignore')

print('Merged shape:', merged.shape)
display(merged.head())

Columnas de df1 tras eliminar irrelevantes: ['Channel', 'Type', 't_mean', 't_std', 't_var', 't_entropy', 't_kurtosis', 't_multiscaleEntropy', 't_time2peak', 't_rms', 't_peak2peak', 't_peak2rms', 't_energy', 't_zcr', 't_PeaksAboveRMSDensity_fun', 'f_peaks_pos_1', 'f_90_percent_energy', 'f_entropy', 'f_mean', 'f_std', 'f_var', 'f_energy', 'f_kurtosis', 'f_multiscaleEntropy', 'f_peak_1020_value', 'f_peak_1020_pos', 'f_peak_2030_value', 'f_peak_2030_pos', 'f_rms', 'f_peak2rms', 'f_power', 'f_PeaksAboveRMSDensity_fun', 'f_peaks_val_2', 'f_peaks_pos_2', 'f_peaks_val_3', 'f_peaks_pos_3', 'w_f_maxval_A6', 'w_f_maxval_D1', 'w_f_maxval_D2', 'w_f_maxval_D3', 'w_f_maxval_D4', 'w_f_maxval_D5', 'w_f_maxval_D6', 'w_f_maxpos_A6', 'w_f_maxpos_D2', 'w_f_maxpos_D3', 'w_f_maxpos_D4', 'w_f_maxpos_D5', 'w_f_maxpos_D6', 'w_f_mean_A6', 'w_f_mean_D1', 'w_f_mean_D2', 'w_f_mean_D3', 'w_f_mean_D4', 'w_f_mean_D5', 'w_f_mean_D6', 'w_t_meanEnergyAD', 'w_t_PEC_A6', 'w_t_PEC_D1', 'w_t_PEC_D2', 'w_t_PEC_D3', 'w_t_PEC_D

,Type,f_90_percent_energy,f_PeaksAboveRMSDensity_fun,f_energy,f_entropy,f_kurtosis,f_mean,f_multiscaleEntropy,f_peak2rms,f_peak_1020_pos,...,w_t_peak2rms_D4,w_t_peak2rms_D5,w_t_peak2rms_D6,w_t_rms_A6,w_t_rms_D1,w_t_rms_D2,w_t_rms_D3,w_t_rms_D4,w_t_rms_D5,w_t_rms_D6
0,EXPL,7.812500,0.064327,88.667131,-6.668203e+06,2.335528,-38.546551,2.105047,8.962248,10.449219,...,3.487185,3.086393,2.744577,0.052869,0.044308,0.118545,0.138755,0.120142,0.097489,0.042294
1,EXPL,7.080078,0.046784,89.716010,-7.224606e+06,1.937270,-40.262434,2.169781,10.486386,10.546875,...,3.607966,4.349338,2.768516,0.043970,0.025376,0.102774,0.168017,0.149922,0.050497,0.025512
2,EXPL,9.960938,0.027290,86.073468,-6.918588e+06,3.035101,-38.394380,1.971289,20.620064,10.156250,...,4.479645,3.081091,3.684229,0.103833,0.056541,0.139821,0.151864,0.083082,0.059533,0.020223
3,EXPL,7.666016,0.050682,54.215575,-7.645442e+06,2.024171,-41.384352,2.156271,8.903901,10.546875,...,3.158368,3.396124,2.617087,0.029720,0.025269,0.120340,0.129489,0.105416,0.035534,0.017943
4,EXPL,9.228516,0.044834,95.018260,-7.519441e+06,2.656138,-39.678304,1.998668,18.769221,10.058594,...,3.581984,3.180676,2.856925,0.090265,0.044063,0.163545,0.159266,0.089725,0.022459,0.016762


In [30]:
# Guardar resultado en formato JSON sin cargar todo en memoria
out_json = workspace / 'merged_datasets.json'
with out_json.open('w', encoding='utf-8') as f:
    f.write('[\n')
    for i, row in enumerate(merged.itertuples(index=False, name=None)):
        record = dict(zip(merged.columns, row))
        json.dump(record, f, ensure_ascii=False, indent=2)
        if i < len(merged) - 1:
            f.write(',\n')
    f.write('\n]\n')
print('Guardado en', out_json)

Guardado en merged_datasets.json


## Notas
- De `df1` se eliminan las columnas `Event_Identifier`, `Station` y `channel` antes de unir.
- En `df2` se renombran las columnas con prefijo `f` seguido de dígitos, por ejemplo `f1_t_mean` a `t_mean`, usando expresiones regulares.
- Los datasets se concatenan verticalmente una vez que las columnas se alinean para evitar merges incorrectos por llave.
- El resultado se guarda en formato JSON estándar como una lista de objetos, escribiendo los registros uno a uno para evitar desbordar la memoria.